# 🚀 Stack 2.9 - Local Training Notebook

**Training on local Mac with MPS (Apple Silicon GPU)**

This notebook trains a LoRA adapter for Stack 2.9 on **Qwen2.5-Coder-7B** using your local machine.

---

**Prerequisites:**
1. Base model downloaded to `./base_model_qwen7b/`
2. Run from the stack-2.9 directory
---


In [ ]:
# Check system and GPU
import platform
import os

print("="*60)
print("SYSTEM INFO")
print("="*60)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"Working directory: {os.getcwd()}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"MPS available: {torch.backends.mps.is_available()}")
    if torch.backends.mps.is_available():
        print(f"MPS device: {torch.backends.mps.is_built()}")
except ImportError:
    print("PyTorch not installed - run: pip install torch")

## 1️⃣ Set Working Directory

In [ ]:
# Navigate to the stack-2.9 directory
import os

STACK_PATH = "/Users/walidsobhi/.openclaw/workspace/stack-2.9"
os.chdir(STACK_PATH)

BASE_MODEL_PATH = "./base_model_qwen7b"
DATA_PATH = "./data/final/train.jsonl"
OUTPUT_DIR = "./training_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Working directory: {os.getcwd()}")
print(f"Base model path: {BASE_MODEL_PATH}")
print(f"Data path: {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

# Check paths exist
print("\n📁 Checking required files:")
print(f"  - Data exists: {os.path.exists(DATA_PATH)}")
if not os.path.exists(DATA_PATH):
    print(f"  ⚠️ Data not found at {DATA_PATH}")

## 2️⃣ Install Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio
!pip install transformers peft accelerate datasets pyyaml tqdm
!pip install scipy
print("\n✅ Dependencies installed")

## 3️⃣ Prepare Training Configuration

In [ ]:
# Use local training config
import yaml

config_path = "./stack/training/train_config_local.yaml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update paths to match our setup
config['model']['name'] = BASE_MODEL_PATH
config['data']['input_path'] = DATA_PATH
config['output']['lora_dir'] = f"{OUTPUT_DIR}/lora"
config['output']['merged_dir'] = f"{OUTPUT_DIR}/merged"
config['hardware']['device'] = "mps"

# Save updated config
updated_config_path = f"{OUTPUT_DIR}/train_config.yaml"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(updated_config_path, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config saved to: {updated_config_path}")
print("\nConfig summary:")
print(f"  - Model: {config['model']['name']}")
print(f"  - Data: {config['data']['input_path']}")
print(f"  - Device: {config['hardware']['device']}")
print(f"  - Epochs: {config['training']['num_epochs']}")

## 4️⃣ Check Base Model

In [ ]:
# Check if base model exists
import os

model_path = BASE_MODEL_PATH
print(f"Checking base model at: {model_path}")

if os.path.exists(model_path):
    files = os.listdir(model_path)
    print(f"✅ Base model found! {len(files)} files:")
    for f in files[:10]:
        print(f"   - {f}")
    if len(files) > 10:
        print(f"   ... and {len(files)-10} more")
else:
    print("❌ Base model NOT found!")
    print("\nTo download Qwen2.5-Coder-7B:")
    print("  huggingface-cli download Qwen/Qwen2.5-Coder-7B --local-dir ./base_model_qwen7b")
    print("  OR")
    print("  python -c \"from transformers import AutoModelForCausalLM; AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-Coder-7B', local_dir='./base_model_qwen7b')\"")

## 5️⃣ Train LoRA Adapter

⚠️ **This will take significant time on MPS.**

MPS is slower than CUDA, so training will take longer. Consider using a smaller dataset for testing.

In [ ]:
import os
import sys

# Add the training module to path
sys.path.insert(0, './stack/training')

print("="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Working directory: {os.getcwd()}")
print(f"Config: {updated_config_path}")
print(f"Output: {OUTPUT_DIR}/lora")
print("="*60 + "\n")

# Run training using the correct function
from train_lora import train_lora

trainer = train_lora(updated_config_path)

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)

## 6️⃣ Verify Training Output

In [ ]:
lora_dir = f"{OUTPUT_DIR}/lora"
print(f"Checking LoRA output: {lora_dir}")

if os.path.exists(lora_dir):
    files = os.listdir(lora_dir)
    print(f"✅ LoRA adapter found! {len(files)} files:")
    for f in files:
        size = os.path.getsize(os.path.join(lora_dir, f)) / (1024*1024)
        print(f"   - {f}: {size:.1f} MB")
else:
    print("⚠️ LoRA directory not found")

## 7️⃣ Merge LoRA Adapter (Optional)

In [ ]:
# Merge LoRA with base model
merged_dir = f"{OUTPUT_DIR}/merged"

print("="*60)
print("MERGING LORA ADAPTER")
print("="*60)

sys.path.insert(0, './stack/training')
from merge_adapter import merge_adapter

# Create a minimal config for merging
import yaml

merge_config = {
    'model': {'name': BASE_MODEL_PATH, 'trust_remote_code': True},
    'output': {'lora_dir': f'{OUTPUT_DIR}/lora', 'merged_dir': merged_dir},
    'quantization': {'enabled': False}
}

# Save config for merge
merge_config_path = f"{OUTPUT_DIR}/merge_config.yaml"
with open(merge_config_path, 'w') as f:
    yaml.dump(merge_config, f)

# Run merge
merge_adapter(
    config_path=merge_config_path,
    lora_path=f"{OUTPUT_DIR}/lora",
    output_path=merged_dir
)

print(f"\n✅ Merged model saved to: {merged_dir}")
if os.path.exists(merged_dir):
    print("Files:", os.listdir(merged_dir)[:5])

## 8️⃣ Test Inference

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = f"{OUTPUT_DIR}/merged"
print(f"Loading model from {model_path}...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="mps",
        trust_remote_code=True
    )
    
    prompt = "Write a Python function to reverse a string:\n\n```python\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("mps")
    
    print("Generating...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("="*40)
    print("RESPONSE:")
    print("="*40)
    print(response[len(prompt):])
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nThis is expected if training hasn't completed yet.")

## 🔚 Training Complete!

Your model is ready in:
- LoRA adapter: `{OUTPUT_DIR}/lora/`
- Merged model: `{OUTPUT_DIR}/merged/`

**Next steps:**
1. Run evaluation on HumanEval/MBPP benchmarks
2. Upload to Hugging Face Hub
3. Apply to Together AI
